### Destroyer

Select `Run All` to tear down this catalog's runtime resources, then drop the catalog.


In [0]:
%pip install databricks-sdk --upgrade

In [0]:
%restart_python

In [ ]:
import sys
sys.path.append('./utils')

CATALOG = dbutils.widgets.get("CATALOG")
print(f"Destroying resources for catalog: {CATALOG}")

state = None
try:
    from uc_state import create_state_manager
    state = create_state_manager(CATALOG)
    print(f"State manager initialized for {CATALOG}")
except Exception as e:
    print(f"⚠️ Could not initialize state manager: {e}")
    print("This usually means the catalog or state table doesn't exist. Skipping resource cleanup.")

In [ ]:
if state:
    try:
        results = state.clear_all()
        for rtype, info in results.items():
            if info["successful"]:
                print(f"✅ {rtype}: deleted {len(info['successful'])} resources")
            if info["failed"]:
                print(f"⚠️ {rtype}: {len(info['failed'])} failures")
                for f in info["failed"]:
                    print(f"   - {f['name']}: {f['reason']}")
    except Exception as e:
        print(f"⚠️ clear_all() failed: {e}")
        print("Continuing with catalog drop...")
else:
    print("No state manager available. Skipping resource cleanup, proceeding to catalog drop.")

In [ ]:
# Belt-and-suspenders sweep BEFORE the catalog drop below.
#
# Why this exists:
#   `state.clear_all()` above only deletes resources tracked in
#   `<CATALOG>._caspers_state.resources`.  The state table itself lives inside
#   <CATALOG>, so any prior `destroy` that half-succeeded — `clear_all()`
#   errored or skipped rows, then the catalog drop wiped the table anyway —
#   leaves orphans with no row anywhere to drive future cleanups.
#
#   Agent serving endpoints, Knowledge Assistants, and Multi-Agent Supervisors
#   all live OUTSIDE Unity Catalog, so they survive `DROP CATALOG CASCADE`
#   indefinitely.  We observed this in practice: after a destroy+redeploy
#   cycle the state table came back empty while 4 serving endpoints and 7
#   KAs/MAS in the `<CATALOG>` namespace were still alive.  The reuse path in
#   `stages/refunder_agent` and `stages/complaint_agent` explicitly skips
#   `add(...)` when an endpoint is reused, so they would never be re-tracked.
#
# What it does:
#   For every serving endpoint whose name starts with `<CATALOG>_` and every
#   tile (KA / MAS) whose name starts with `<CATALOG>-`, attempt deletion.
#   Best-effort: 'already-gone' and any other error is logged but does not
#   block the catalog drop downstream.
#
# Naming conventions assumed (current in this repo):
#   - serving endpoints:  `<CATALOG>_<purpose>_agent`  e.g. `oleksandra_refund_agent`
#   - KA / MAS tiles:     `<CATALOG>-<purpose>`        e.g. `oleksandra-legal`
# If those conventions change, update the prefix tests below.

from databricks.sdk import WorkspaceClient

_w = WorkspaceClient()
_CAT_LOWER = CATALOG.lower()


def _is_not_found(e):
    if type(e).__name__ in {"NotFound", "ResourceDoesNotExist"}:
        return True
    m = str(e).lower()
    return "does not exist" in m or "not found" in m or "no such" in m or "404" in m


print(f"\n🔎 Sweep: serving endpoints matching '{_CAT_LOWER}_*'")
for ep in _w.serving_endpoints.list():
    name = (ep.name or "")
    if not name.lower().startswith(f"{_CAT_LOWER}_"):
        continue
    try:
        _w.serving_endpoints.delete(name=name)
        print(f"  ✅ deleted endpoint {name}")
    except Exception as e:
        if _is_not_found(e):
            print(f"  ↪︎ already gone     {name}")
        else:
            print(f"  ⚠️  {name}: {type(e).__name__}: {e}")

print(f"\n🔎 Sweep: tiles (KA / MAS) matching '{_CAT_LOWER}-*'")
try:
    _tiles_res = _w.api_client.do("GET", "/api/2.0/tiles")
    _tiles = _tiles_res.get("tiles", []) if isinstance(_tiles_res, dict) else []
except Exception as e:
    print(f"  ⚠️  tiles.list failed: {type(e).__name__}: {e}")
    _tiles = []

for _t in _tiles:
    _name = (_t.get("name") or "")
    if not _name.lower().startswith(f"{_CAT_LOWER}-"):
        continue
    _tile_id = _t.get("id") or _t.get("tile_id")
    _kind = (_t.get("tile_type") or _t.get("type") or "").upper()
    # Same path precedence uc_state.clear_all() uses: typed APIs first, then
    # Tiles API as the legacy fallback.
    _paths = (
        ["/api/2.1/knowledge-assistants", "/api/2.0/knowledge-assistants", "/api/2.0/tiles"]
        if _kind == "KA" else
        ["/api/2.0/multi-agent-supervisors", "/api/2.0/tiles"]
    )
    _deleted = False
    _last_err = None
    for _p in _paths:
        try:
            _w.api_client.do("DELETE", f"{_p}/{_tile_id}")
            print(f"  ✅ deleted {_kind or '???':3} {_name!r} via {_p}")
            _deleted = True
            break
        except Exception as e:
            _last_err = e
    if not _deleted:
        print(f"  ⚠️  {_name!r}: tried all paths, last err: {type(_last_err).__name__}: {_last_err}")

# Demo-created UC SQL functions (Act 2 / Genie Code) in ai are
# NOT deployed by stages/catastrophe_command.ipynb — operators create them live
# during the demo. Drop explicitly before the catalog CASCADE below so cleanup
# is idempotent even when the catalog is kept or the function was never created.
_DEMO_UC_FUNCTIONS = [
    "active_crossing_status",
]
_UC_SCHEMA = f"{CATALOG}.ai"


def _sql_gone(e):
    if _is_not_found(e):
        return True
    msg = str(e).lower()
    return any(s in msg for s in (
        "does not exist", "not found", "cannot find", "unknown function",
    ))


print(f"\n🔎 Sweep: demo-created UC functions in {_UC_SCHEMA}")
for _fn in _DEMO_UC_FUNCTIONS:
    _stmt = f"DROP FUNCTION IF EXISTS {_UC_SCHEMA}.{_fn}"
    try:
        spark.sql(_stmt)
        print(f"  ✅ dropped {_UC_SCHEMA}.{_fn}")
    except Exception as e:
        if _sql_gone(e):
            print(f"  ↪︎ already gone     {_UC_SCHEMA}.{_fn}")
        else:
            print(f"  ⚠️  {_UC_SCHEMA}.{_fn}: {type(e).__name__}: {e}")


In [0]:
%sql
DROP CATALOG IF EXISTS IDENTIFIER(:CATALOG) CASCADE;